In [1]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

# Root path to dataset
DATA_DIR = Path("/home/rishi/medai/data/chest-xray-pneumonia/chest-pneumonia/archive/chest_xray/chest_xray")

# Use GPU if available, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5070


In [2]:
# Training transforms include augmentation to improve generalization
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    # ImageNet mean/std since we're using pretrained ResNet18
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Eval transforms are deterministic — no augmentation during validation or test
eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [3]:
# Load the train folder twice so each subset gets the correct transform
full_train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=train_tfms)
full_eval_ds  = datasets.ImageFolder(DATA_DIR / "train", transform=eval_tfms)

# 80/20 split with fixed seed for reproducibility
train_size = int(0.8 * len(full_train_ds))
indices    = torch.randperm(len(full_train_ds), generator=torch.Generator().manual_seed(42)).tolist()

train_indices = indices[:train_size]
val_indices   = indices[train_size:]

# Subset each dataset with the correct transform
train_ds = Subset(full_train_ds, train_indices)
val_ds   = Subset(full_eval_ds,  val_indices)

print("Train size:", len(train_ds))
print("Val size:",   len(val_ds))
print("Classes:", full_train_ds.classes)
print("Class mapping:", full_train_ds.class_to_idx)

Train size: 4172
Val size: 1044
Classes: ['NORMAL', 'PNEUMONIA']
Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}


In [6]:
# Shuffle train each epoch; val order doesn't matter
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print("Train batches:", len(train_loader))
print("Val batches:",   len(val_loader))

Train batches: 131
Val batches: 33


In [7]:
# Load pretrained ResNet18 and replace final layer for binary classification
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 1)  # Single output logit
model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=1, bias=True)


In [8]:
# Compensate for class imbalance (~3:1 pneumonia:normal) using pos_weight
pos_weight = torch.tensor([3875 / 1341]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Adam with low lr to avoid destroying pretrained weights
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-4)

print("pos_weight:", pos_weight.item())

pos_weight: 2.889634609222412


In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()  # Enables dropout and batchnorm training behavior
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)  # Match output shape (batch, 1)

        optimizer.zero_grad()       # Clear gradients from previous step
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()             # Compute gradients
        optimizer.step()            # Update weights

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()  # Convert logit to binary prediction
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()  # Disables dropout and batchnorm training behavior
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():  # No gradients needed during evaluation
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total